# apertus-eval-prep — paper-matrix **vLLM backend** (Colab ID-5)

[Open in Colab](https://colab.research.google.com/github/Shivani767/apertus-eval-prep/blob/master/notebooks/colab_stability_backend.ipynb)

Runtime → **T4 GPU**.

**If you already hit a `libcudart` / `Config(deprecated=…)` error:** Runtime → **Disconnect and delete runtime**, then reconnect (T4). A half-upgraded torch cannot be fixed by re-running cells.

Do **not** Run all. Every session: **cell 1 → cell 2 → cell 3 (Drive) → one sweep**.

**Install rule:** install official **`vllm==0.24.0+cu129`** (matches Colab `torch==2.11`). Never bare `pip install vllm` (pulls 0.27 + torch 2.13 / CUDA 13).

Drive: `MyDrive/apertus-eval-prep-paper`.


In [ ]:
# Cell 1 — clone / pull (no vLLM)
import os
if os.path.exists("pyproject.toml") and os.path.exists("src/apertus_eval_prep"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[viz]"
!git log -1 --oneline


In [ ]:
# Cell 2 — install vLLM 0.24.0+cu129 (matches Colab torch 2.11)
import os, sys, subprocess
from pathlib import Path

if Path("pyproject.toml").exists() and Path("src/apertus_eval_prep").exists():
    pass
elif Path("apertus-eval-prep/pyproject.toml").exists():
    os.chdir("apertus-eval-prep")
else:
    raise FileNotFoundError("Run cell 1 first.")

!git pull --ff-only
!pip -q install -e ".[viz]"
!git log -1 --oneline

_repo_src = str((Path.cwd() / "src").resolve())
if _repo_src not in sys.path:
    sys.path.insert(0, _repo_src)

import torch
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0), "torch", torch.__version__, "cuda", torch.version.cuda)
if not str(torch.__version__).startswith("2.11"):
    raise RuntimeError(
        f"Expected Colab torch 2.11.x, got {torch.__version__}. "
        "Runtime → Disconnect and delete runtime, then reconnect."
    )

VLLM_VER = "0.24.0"
VLLM_WHEEL = (
    f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VER}/"
    f"vllm-{VLLM_VER}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
)
TORCH_INDEX = "https://download.pytorch.org/whl/cu128"

def _pip(*args):
    cmd = [sys.executable, "-m", "pip", *args]
    print("+", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.stdout.strip():
        print(p.stdout[-1200:], flush=True)
    if p.returncode != 0:
        print(p.stderr[-2000:], flush=True)
        raise RuntimeError(f"pip failed ({p.returncode})")

# Remove any broken PyPI / 0.27 install left from earlier attempts.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"], check=False)

# --no-deps: do not let pip replace Colab torch with a different build.
_pip("install", "-q", "--no-deps", VLLM_WHEEL)

# Non-torch deps declared by vllm 0.24 (torch stays Colab's 2.11.0+cu128).
_pip(
    "install", "-q",
    "transformers>=4.56.0",
    "tokenizers>=0.21.1",
    "sentencepiece",
    "protobuf",
    "fastapi",
    "uvicorn[standard]",
    "openai",
    "prometheus_client",
    "prometheus-fastapi-instrumentator",
    "lm-format-enforcer>=0.10.11",
    "outlines_core==0.2.11",
    "xgrammar",
    "llguidance",
    "gguf",
    "mistral_common>=1.8.8",
    "compressed-tensors",
    "depyf",
    "cloudpickle",
    "watchfiles",
    "python-json-logger",
    "einops",
    "importlib_metadata",
    "partial_json_parser",
    "pyzmq",
    "msgspec",
    "blake3",
    "pybase64",
    "pillow",
    "tiktoken",
    "huggingface_hub",
    "aiohttp",
    "filelock",
    "psutil",
    "ray>=2.48.0",
    "ninja",
)

# Matching vision/audio builds for torch 2.11 (optional but keeps imports clean).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torchvision==0.26.0+cu128", "torchaudio==2.11.0+cu128",
     "--index-url", TORCH_INDEX],
    check=False,
)

for name in list(sys.modules):
    if name == "vllm" or name.startswith("vllm."):
        del sys.modules[name]

from vllm import LLM, SamplingParams  # noqa: F401
import vllm
import torch as _t
print("vllm", getattr(vllm, "__version__", "?"), "torch", _t.__version__)
assert str(_t.__version__).startswith("2.11"), _t.__version__

if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")


In [ ]:
# Cell 3 — Drive + sweep helper
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run cells 1–2 first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
print("partials:", len(list(Path("results/runs").glob("*.partial.jsonl"))), flush=True)

def save_paper():
    import subprocess
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]; i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]; i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    from vllm import LLM  # noqa: F401
    from apertus_eval_prep.sweep import execute_sweep
    print(f"sweep model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    print({"n_cells": len(planned), "n_skip": sum(1 for p in planned if p["skipped"])}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --only-factor backend --out-dir results/runs --registry results/registry_paper.jsonl | head -n 20


## ID-5 — one model per session

T4 skips Qwen-7B vLLM.


In [ ]:
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "backend")


In [ ]:
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "backend")


In [ ]:
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "backend")


Unpack the Drive zip on Mac. Commit `results/runs/*.json` and `results/registry_paper.jsonl`. Do not edit numbers.
